In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr

import os
import numpy as np
from tqdm import tqdm

from scipy.interpolate import griddata
from scipy.spatial import distance
from pyproj import CRS, Proj, Transformer
from shapely.geometry import box

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import time

import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Change from 5km to 1km to get 1km predictions
regridded_files = ['PM_AOD/5km_grid_features/'+f 
                       for f in os.listdir('PM_AOD/5km_grid_features/')]
regridded_files.sort()
df_all = []

for f in regridded_files:
    data = xr.open_dataset(f)    
    df = data.to_dataframe().reset_index()   
    df = df.dropna(subset=['aod','stations','u10','v10','d2m','t2m','sp','tp','blh']).drop(columns=['x','y','Projected Y','Projected X','stations_interpolated'])    
    df['year'], df['month'], df['day'] = f[-13:-9], f[-8:-6], f[-5:-3]    
    df = df.reset_index(drop=True)   
    df = df.fillna(0)   
    df_all.append(df)

data_no_na = pd.concat(df_all)
covariates = list(set(data_no_na.columns.tolist()) - set(['stations']))
covariates.sort()

In [20]:
data = data_no_na.copy()

lon_bins = pd.cut(data['lon'], bins=8, labels=False)
lat_bins = pd.cut(data['lat'],  bins=8, labels=False)
data['sp_block'] = (lat_bins.astype(int) * 100 + lon_bins.astype(int)).astype(int)  

X_cols = covariates               
y_col  = 'stations'

In [21]:
# Unique spatiotemporal groups 
grp = (
    data.groupby(['year','sp_block'])
        .size()
        .reset_index(name='n_rows')      
        .sort_values('n_rows', ascending=False)
        .reset_index(drop=True)
)

K = min(10, len(grp))                     
fold_load = np.zeros(K, dtype=int)        
fold_id   = np.full(len(grp), -1, dtype=int)

for i in range(len(grp)):
    k = int(fold_load.argmin())
    fold_id[i] = k
    fold_load[k] += int(grp.loc[i, 'n_rows'])

grp['fold'] = fold_id

data = data.merge(grp[['year','sp_block','fold']], on=['year','sp_block'], how='left')

print("Fold loads (rows per fold):", dict(zip(range(K), fold_load)))
print(grp.groupby('fold').agg(groups=('sp_block','count'),
                              years=('year', 'nunique'),
                              rows=('n_rows','sum')).reset_index())

Fold loads (rows per fold): {0: 82025, 1: 82018, 2: 82013, 3: 82020, 4: 82019, 5: 82021, 6: 82019, 7: 82018, 8: 82014, 9: 82020}
   fold  groups  years   rows
0     0      95     20  82025
1     1      94     20  82018
2     2      96     20  82013
3     3      94     20  82020
4     4      94     18  82019
5     5      94     20  82021
6     6      94     20  82019
7     7      94     20  82018
8     8      95     20  82014
9     9      94     19  82020


In [22]:
def eval_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

def make_train_es_split(X_train_full, y_train_full, groups_blocks, test_size=0.1, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, es_idx = next(gss.split(X_train_full, y_train_full, groups=groups_blocks))
    return X_train_full[tr_idx], y_train_full[tr_idx], X_train_full[es_idx], y_train_full[es_idx]

def run_xgb(X_tr, y_tr, X_es, y_es, X_val, y_val):

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_es, label=y_es)
    dtest  = xgb.DMatrix(X_val)

    params = {
        "objective": "reg:squarederror",
        "eta": 0.05,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "tree_method": "hist",
        "eval_metric": "rmse",
        "seed": 42
    }

    bst = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_ntree_limit = getattr(bst, "best_ntree_limit", None)
    y_hat = bst.predict(dtest, ntree_limit=best_ntree_limit) if best_ntree_limit else bst.predict(dtest)
    rmse, mae, r2 = eval_metrics(y_val, y_hat)

    return {
        'RMSE': rmse, 'MAE': mae, 'R2': r2,
        'model': 'XGBoost',
        'best_iter': int(getattr(bst, "best_iteration", 0)),
        'best_ntree_limit': int(best_ntree_limit) if best_ntree_limit else None
    }

def run_lgbm(X_tr, y_tr, X_es, y_es, X_val, y_val):

    lgb_train = lgb.Dataset(X_tr, label=y_tr)
    lgb_valid = lgb.Dataset(X_es, label=y_es, reference=lgb_train)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 255,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "lambda_l2": 1.0,
        "lambda_l1": 0.0,
        "verbosity": -1,
        "seed": 42
    }

    booster = lgb.train(
        params=params,
        train_set=lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_valid],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(0)]
    )

    y_hat = booster.predict(X_val, num_iteration=booster.best_iteration)
    rmse, mae, r2 = eval_metrics(y_val, y_hat)
    return {
        'RMSE': rmse, 'MAE': mae, 'R2': r2,
        'model': 'LightGBM',
        'best_iter': int(booster.best_iteration)
    }

def run_rf(X_tr_full, y_tr_full, X_val, y_val):
    rf = RandomForestRegressor(
        n_estimators=800,
        max_depth=None,
        min_samples_leaf=1,
        max_features='sqrt',
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_tr_full, y_tr_full)
    y_hat = rf.predict(X_val)
    rmse, mae, r2 = eval_metrics(y_val, y_hat)
    return {
        'RMSE': rmse, 'MAE': mae, 'R2': r2,
        'model': 'RandomForest',
        'n_trees': int(rf.n_estimators)
    }

In [23]:
records = []
algos = ["xgb", "lgbm", "rf"]  
print(f"\nStarting spatiotemporal CV: K={K} folds "
      f"| rows={len(data):,} | features={len(X_cols)}")

for fold in range(K):
    val_mask   = (data['fold'] == fold)
    train_mask = ~val_mask

    X_train_full = data.loc[train_mask, X_cols].astype(float).values
    y_train_full = data.loc[train_mask, y_col].astype(float).values
    X_val        = data.loc[val_mask,   X_cols].astype(float).values
    y_val        = data.loc[val_mask,   y_col].astype(float).values

    if X_val.shape[0] < 100 or X_train_full.shape[0] < 1000:
        print(f"[fold {fold}] skipped (train={len(y_train_full)}, val={len(y_val)})")
        continue

    groups_train_blocks = data.loc[train_mask, 'sp_block'].values
    X_tr, y_tr, X_es, y_es = make_train_es_split(X_train_full, y_train_full, groups_train_blocks)

    print(f"\n=== Fold {fold} ===  train={len(y_train_full):,}  val={len(y_val):,}  "
          f"years_in_val={data.loc[val_mask,'year'].nunique()}  "
          f"blocks_in_val={data.loc[val_mask,'sp_block'].nunique()}")

    for algo in algos:
        t0 = time.time()
        if algo == "xgb":
            out = run_xgb(X_tr, y_tr, X_es, y_es, X_val, y_val)
        elif algo == "lgbm":
            out = run_lgbm(X_tr, y_tr, X_es, y_es, X_val, y_val)
        elif algo == "rf":
            out = run_rf(X_train_full, y_train_full, X_val, y_val)
        else:
            continue

        out.update({
            'model': out.get('model', algo.upper()),
            'fold': int(fold),
            'val_rows': int(len(y_val)),
            'train_rows': int(len(y_train_full)),
            'years_in_val': int(data.loc[val_mask,'year'].nunique()),
            'blocks_in_val': int(data.loc[val_mask,'sp_block'].nunique()),
            'secs': round(time.time() - t0, 3)
        })
        records.append(out)

        print(f"[{out['model']}] fold={fold}  RMSE={out['RMSE']:.3f}  "
              f"MAE={out['MAE']:.3f}  R2={out['R2']:.3f}  "
              f"(val_rows={out['val_rows']}, train_rows={out['train_rows']})")

cv_df = pd.DataFrame.from_records(records)
print("\nPer-fold summary (mean by model):\n",
      cv_df.groupby('model')[['RMSE','MAE','R2']].mean().round(3))


Starting spatiotemporal CV: K=10 folds | rows=820,187 | features=245

=== Fold 0 ===  train=738,162  val=82,025  years_in_val=20  blocks_in_val=45
[XGBoost] fold=0  RMSE=3.366  MAE=2.323  R2=0.749  (val_rows=82025, train_rows=738162)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2321]	valid_0's rmse: 3.31003
[LightGBM] fold=0  RMSE=3.161  MAE=2.172  R2=0.779  (val_rows=82025, train_rows=738162)
[RandomForest] fold=0  RMSE=4.530  MAE=2.916  R2=0.545  (val_rows=82025, train_rows=738162)

=== Fold 1 ===  train=738,169  val=82,018  years_in_val=20  blocks_in_val=43
[XGBoost] fold=1  RMSE=3.369  MAE=2.334  R2=0.733  (val_rows=82018, train_rows=738169)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1654]	valid_0's rmse: 3.35589
[LightGBM] fold=1  RMSE=3.161  MAE=2.189  R2=0.765  (val_rows=82018, train_rows=738169)
[RandomForest] fold=1  RMSE=4.444  MAE=2.966  R2=0.535  (val_rows=82018, tra